# TFT sanity notebook (robust)

This notebook loads the prepared TFT input parquets, builds a `TimeSeriesDataSet` using `src.configs.tft_v1` **if available**, otherwise falls back to inference from columns. It then loads a TFT checkpoint and runs a prediction sanity check (RMSE/MAE).

In [ ]:
# --- imports ---
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

# PyTorch Forecasting
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer

pd.set_option("display.max_columns", 200)
torch.set_float32_matmul_precision("high")


## Paths
Set your repo root and parquet paths. These are your current defaults.

In [ ]:
REPO_ROOT = Path.home() / "pv_forecast_30d"
sys.path.insert(0, str(REPO_ROOT))

train_p = REPO_ROOT / "data/processed/pretraining/germany/global/tft_inputs/regional_train_tft_full.parquet"
val_p   = REPO_ROOT / "data/processed/pretraining/germany/global/tft_inputs/regional_val_tft_full.parquet"

assert train_p.exists(), f"Missing: {train_p}"
assert val_p.exists(), f"Missing: {val_p}"

print("train:", train_p)
print("val  :", val_p)


## Load data

In [ ]:
train = pd.read_parquet(train_p)
val   = pd.read_parquet(val_p)

print("train shape:", train.shape)
print("val shape  :", val.shape)
display(train.head(3))


## Basic fixes
- Ensure `timestamp_utc` is UTC tz-aware
- Ensure `time_idx` exists (create from timestamp per plant if missing)
- Drop columns specified in `src.configs.tft_v1.DROP_COLS` if present

In [ ]:
# timestamp
if "timestamp_utc" in train.columns:
    train["timestamp_utc"] = pd.to_datetime(train["timestamp_utc"], utc=True, errors="coerce")
if "timestamp_utc" in val.columns:
    val["timestamp_utc"] = pd.to_datetime(val["timestamp_utc"], utc=True, errors="coerce")

# import config module (optional)
try:
    import src.configs.tft_v1 as cfg
    print("Loaded cfg from:", cfg.__file__)
except Exception as e:
    cfg = None
    print("Could not import src.configs.tft_v1:", repr(e))

# drop cols from cfg if available
drop_cols = []
if cfg is not None and hasattr(cfg, "DROP_COLS"):
    drop_cols = list(getattr(cfg, "DROP_COLS"))
for c in drop_cols:
    if c in train.columns:
        train = train.drop(columns=[c])
    if c in val.columns:
        val = val.drop(columns=[c])

# ensure group column exists
group_col = "plant_id" if "plant_id" in train.columns else None
if cfg is not None and hasattr(cfg, "GROUP_COL"):
    # some configs store as string
    group_col = getattr(cfg, "GROUP_COL")
if group_col is None:
    raise KeyError("Could not find group column. Expected plant_id or cfg.GROUP_COL")

# ensure time_idx
if "time_idx" not in train.columns:
    if "timestamp_utc" not in train.columns:
        raise KeyError("Need either time_idx or timestamp_utc to build time index")
    train = train.sort_values([group_col, "timestamp_utc"]).copy()
    train["time_idx"] = train.groupby(group_col).cumcount()
if "time_idx" not in val.columns:
    if "timestamp_utc" not in val.columns:
        raise KeyError("Need either time_idx or timestamp_utc to build time index")
    val = val.sort_values([group_col, "timestamp_utc"]).copy()
    val["time_idx"] = val.groupby(group_col).cumcount()

print("Columns (train):", len(train.columns))
print("Has time_idx:", "time_idx" in train.columns, "Has timestamp_utc:", "timestamp_utc" in train.columns)


## Feature-role lists
Prefer `cfg` lists when present. Otherwise infer reasonable defaults.

Rules for fallback:
- `target` = `power_norm` if present else first column matching `*power*`
- categoricals = `[plant_id]` plus `weather_code` if present
- known reals = numeric columns that are not target and not obvious encodings
- unknown reals = columns that look like LSTM encodings (`lstm*`, `enc*`, `embedding*`) and optionally the target itself (TFT handles target separately)

In [ ]:
TARGET = "power_norm" if "power_norm" in train.columns else None
if cfg is not None and hasattr(cfg, "TARGET_COL"):
    TARGET = getattr(cfg, "TARGET_COL")
if TARGET is None:
    # crude fallback: first column containing 'power'
    power_cols = [c for c in train.columns if "power" in c.lower()]
    if not power_cols:
        raise KeyError("Could not infer target column. Expected power_norm or cfg.TARGET_COL.")
    TARGET = power_cols[0]

TIME = "time_idx"
GROUP = [group_col]

def _present(cols):
    return [c for c in cols if c in train.columns]

# categoricals
static_categoricals = ["plant_id"] if "plant_id" in train.columns else [group_col]
if cfg is not None and hasattr(cfg, "STATIC_CATEGORICALS"):
    static_categoricals = list(getattr(cfg, "STATIC_CATEGORICALS"))
static_categoricals = _present(static_categoricals)

tv_known_categoricals = []
if cfg is not None and hasattr(cfg, "TV_KNOWN_CATEGORICALS"):
    tv_known_categoricals = list(getattr(cfg, "TV_KNOWN_CATEGORICALS"))
tv_known_categoricals = _present(tv_known_categoricals)

# numeric feature lists
tv_known_reals = []
tv_unknown_reals = []
if cfg is not None and hasattr(cfg, "TV_KNOWN_REALS"):
    tv_known_reals = list(getattr(cfg, "TV_KNOWN_REALS"))
if cfg is not None and hasattr(cfg, "TV_UNKNOWN_REALS"):
    tv_unknown_reals = list(getattr(cfg, "TV_UNKNOWN_REALS"))

if tv_known_reals and tv_unknown_reals:
    tv_known_reals = _present(tv_known_reals)
    tv_unknown_reals = _present(tv_unknown_reals)
else:
    # infer
    num_cols = [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c])]
    # remove identifiers
    num_cols = [c for c in num_cols if c not in {TIME, TARGET}]
    # encoding-ish columns
    enc = [c for c in num_cols if c.lower().startswith(("lstm", "enc", "embedding")) or "encoding" in c.lower()]
    tv_unknown_reals = enc
    tv_known_reals = [c for c in num_cols if c not in enc]

# optional: weather_code as categorical if exists (it is numeric in parquet usually)
if "weather_code" in train.columns and "weather_code" not in static_categoricals and "weather_code" not in tv_known_categoricals:
    tv_known_categoricals = ["weather_code"] + tv_known_categoricals

print("TARGET:", TARGET)
print("GROUP :", GROUP)
print("TIME  :", TIME)
print("static_categoricals:", static_categoricals)
print("tv_known_categoricals:", tv_known_categoricals)
print("tv_known_reals (n):", len(tv_known_reals))
print("tv_unknown_reals (n):", len(tv_unknown_reals))

missing_from_cfg = []
if cfg is not None and hasattr(cfg, "TV_KNOWN_REALS"):
    missing_from_cfg = [c for c in getattr(cfg, "TV_KNOWN_REALS") if c not in train.columns]
    if missing_from_cfg:
        print("Missing TV_KNOWN_REALS columns in parquet (showing up to 30):", missing_from_cfg[:30])


## Cast categoricals (fix for `weather_code` numeric error)
PyTorch Forecasting requires categoricals to be string/object/category, not numeric.

In [ ]:
cat_cols = []
cat_cols += static_categoricals
cat_cols += tv_known_categoricals
cat_cols += GROUP
cat_cols = [c for c in dict.fromkeys(cat_cols) if c in train.columns]

def cast_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in cat_cols:
        df[c] = df[c].astype("string").fillna("UNK").astype("category")
    return df

train = cast_categoricals(train)
val   = cast_categoricals(val)

print({c: str(train[c].dtype) for c in cat_cols})


## Load TFT checkpoint
Set `CKPT_PATH` or let the notebook auto-search in your repo under `experiments/tft/runs`.

If auto-search finds multiple checkpoints, it picks the newest by modified time.

In [ ]:
# Option A: set manually
CKPT_PATH = None  # e.g., REPO_ROOT / "experiments/tft/runs/germany/v1.0/fold_0/checkpoints/last.ckpt"

# Option B: auto-search
if CKPT_PATH is None:
    candidates = list((REPO_ROOT / "experiments").rglob("*.ckpt"))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        raise FileNotFoundError("No .ckpt found under REPO_ROOT/experiments. Set CKPT_PATH manually.")
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    CKPT_PATH = candidates[0]

print("Using CKPT:", CKPT_PATH)

tft = TemporalFusionTransformer.load_from_checkpoint(str(CKPT_PATH))
tft.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
tft.to(device)
print("Model device:", next(tft.parameters()).device)


## Encoder and prediction lengths
If you trained TFT with certain lengths, set them here.

Defaults below are conservative for 15-min data:
- encoder = 96 (1 day)
- prediction = 96 (1 day)

Change them to match your training run if needed.

In [ ]:
# Prefer lengths and loader params from the checkpoint if available.
# If not available, fall back to conservative defaults.

def _hp_get(model, key, default):
    for attr in ("hparams", "hyper_parameters"):
        if hasattr(model, attr):
            hp = getattr(model, attr)
            try:
                if isinstance(hp, dict) and key in hp:
                    return hp[key]
            except Exception:
                pass
    return default

# Compute minimal series length per group to avoid impossible window sizes
min_len = train.groupby(GROUP[0])[TIME].nunique().min()
print("Min timesteps per group:", int(min_len))

max_encoder_length = int(_hp_get(tft, "max_encoder_length", 96))
max_prediction_length = int(_hp_get(tft, "max_prediction_length", 96))

# Clamp if your data is shorter than the requested window
# Need at least encoder + prediction steps.
if min_len <= max_encoder_length + max_prediction_length:
    # keep a usable split
    max_prediction_length = max(1, int(min_len // 4))
    max_encoder_length = max(1, int(min_len - max_prediction_length - 1))
    print("Clamped lengths due to short series.")

batch_size = int(_hp_get(tft, "batch_size", 128))
num_workers = int(_hp_get(tft, "num_workers", 4))

print("Using:", {"max_encoder_length": max_encoder_length,
               "max_prediction_length": max_prediction_length,
               "batch_size": batch_size,
               "num_workers": num_workers})


## Build TimeSeriesDataSet and DataLoaders

In [ ]:
train_ds = TimeSeriesDataSet(
    train,
    time_idx=TIME,
    target=TARGET,
    group_ids=GROUP,
    min_encoder_length=max_encoder_length,
    max_encoder_length=max_encoder_length,
    min_prediction_length=max_prediction_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=static_categoricals,
    time_varying_known_categoricals=tv_known_categoricals,
    time_varying_known_reals=tv_known_reals,
    time_varying_unknown_reals=tv_unknown_reals,
    target_normalizer=None,                # power_norm already normalized 0..1
    add_relative_time_idx=True,
    add_target_scales=False,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

val_ds = TimeSeriesDataSet.from_dataset(train_ds, val, predict=True, stop_randomization=True)

train_dl = train_ds.to_dataloader(train=True, batch_size=batch_size, num_workers=num_workers)
val_dl   = val_ds.to_dataloader(train=False, batch_size=batch_size, num_workers=num_workers)

print("batches:", len(train_dl), len(val_dl))


## Predict and compute metrics (robust)
Handles different return types across `pytorch-forecasting` versions and moves tensors to CPU before converting to NumPy.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def _unwrap_y(y):
    # y can be Tensor, (y, weight), list/tuple etc.
    if isinstance(y, (list, tuple)):
        # common: (y, weight)
        if len(y) >= 1:
            return y[0]
    return y

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

pred = tft.predict(val_dl, mode="prediction", return_index=True, return_y=True)

# normalize outputs
index = None
if isinstance(pred, tuple) and len(pred) >= 1:
    y_hat = pred[0]
    if len(pred) >= 2:
        index = pred[1]
    y_true = pred[2] if len(pred) >= 3 else None
else:
    y_hat = pred.output if hasattr(pred, "output") else pred
    index = getattr(pred, "index", None)
    y_true = getattr(pred, "y", None)

y_true = _unwrap_y(y_true)

y_hat = to_numpy(y_hat)
y_true = to_numpy(y_true)

print("y_hat:", y_hat.shape, "y_true:", y_true.shape)
if index is None:
    print("index: None")
else:
    try:
        display(index.head())
    except Exception:
        print(index)

# flatten and score
y_hat_flat = y_hat.reshape(-1)
y_true_flat = y_true.reshape(-1)

rmse = mean_squared_error(y_true_flat, y_hat_flat, squared=False)
mae  = mean_absolute_error(y_true_flat, y_hat_flat)

print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
